# [SQL 재현] 소방청 화재발생 통계 × 인구 데이터 분석

## 단계: 04. 화재 집계·JOIN — SQL 재현
- 목표: 화재 원자료(사건 단위)를 SQL로 정제·집계하고, 인구 테이블과 JOIN해 10만 명당 화재발생율을 만든 뒤 Python·SAS 결과와 대조한다.
- 환경: Jupyter Notebook + sqlite3 (sql_practice.db에 화재·인구 테이블 추가)
- 대조 기준: nfa_fire_2024.ipynb (1.3-1.5), nfa_fire_2024.sas (1.2-1.4, PROC SQL 41~71행)
- 이번 범위: 적재 → 정제(결측 대체·중복 제거) → 시군구 집계 → LEFT JOIN → 발생율. 비율 변수·지역유형 분석은 이후 단계.

### 4.1 데이터 적재
#### 4.1-1 CSV 2개를 SQLite 테이블로 저장
- 화재 정제본(fire_clean_for_sas.csv) → `fire_raw`, 인구 요약(population_sigungu_avg.csv) → `pop`
- 두 파일 모두 utf-8-sig 인코딩, 영문 컬럼명 (SAS 연동 때 만든 파일)

In [1]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(r'C:\data\sql_practice.db')

fire_raw = pd.read_csv(r'C:\data\fire_clean_for_sas.csv', encoding='utf-8-sig')
pop      = pd.read_csv(r'C:\data\population_sigungu_avg.csv', encoding='utf-8-sig')

fire_raw.to_sql('fire_raw', conn, if_exists='replace', index=False)
pop.to_sql('pop', conn, if_exists='replace', index=False)

265

#### 4.1-2 화재 원자료 컬럼 구성 확인
- SQL_01 1.2-4와 같은 방식 (`PRAGMA table_info`)
- 4.2-1에서 컬럼 이름 16개를 직접 적어야 하므로 이름을 먼저 확인한다

In [2]:
pd.read_sql("PRAGMA table_info(fire_raw)", conn)

,cid,name,type,notnull,dflt_value,pk
0,0,FIRE_DT,TEXT,0,None,0
1,1,SIDO,TEXT,0,None,0
2,2,SIGUNGU,TEXT,0,None,0
3,3,FIRE_TYPE,TEXT,0,None,0
4,4,CAUSE_L,TEXT,0,None,0
5,5,CAUSE_S,TEXT,0,None,0
6,6,CASUALTY,INTEGER,0,None,0
7,7,DEATH,INTEGER,0,None,0
8,8,INJURY,INTEGER,0,None,0
9,9,DMG_TOTAL,INTEGER,0,None,0


> **컬럼 16개 확인 — 문자(TEXT) 9개, 정수(INTEGER) 5개, 소수(REAL) 2개** <br>
> 소수형은 DMG_REAL·DMG_MOVABLE 두 열이며, 4.1-3에서 결측이 있는 것으로 확인된 두 열과 같다.

#### 4.1-3 행 수·결측 확인
- SAS 1.2 `proc means n nmiss` 대응 (CASUALTY·DEATH·INJURY·DMG 3개 열)
- 인구 테이블 행 수도 함께 확인

In [3]:
q = """
SELECT COUNT(*) AS 행수,
       COUNT(*) - COUNT(CASUALTY) AS CASUALTY_결측,
       COUNT(*) - COUNT(DEATH) AS DEATH_결측,
       COUNT(*) - COUNT(INJURY) AS INJURY_결측,
       COUNT(*) - COUNT(DMG_TOTAL) AS DMG_TOTAL_결측,
       COUNT(*) - COUNT(DMG_REAL) AS DMG_REAL_결측,
       COUNT(*) - COUNT(DMG_MOVABLE) AS DMG_MOVABLE_결측,
       (SELECT COUNT(*) FROM pop) AS 인구_행수
FROM fire_raw
"""
pd.read_sql(q, conn)

,행수,CASUALTY_결측,DEATH_결측,INJURY_결측,DMG_TOTAL_결측,DMG_REAL_결측,DMG_MOVABLE_결측,인구_행수
0,191509,0,0,0,0,15070,15070,265


> **화재 정제본 191,509행, 인구 265행** <br>
> 결측: CASUALTY·DEATH·INJURY·DMG_TOTAL 0, DMG_REAL·DMG_MOVABLE 각 15,070 (전체의 약 7.9%). <br>
> 인구 265행 — 노트북 1.4-2와 일치.

### 4.2 정제·집계
#### 4.2-1 결측 대체 + 완전 중복 행 제거 → fire 테이블
- SAS 1.2 대응: DMG_REAL·DMG_MOVABLE 결측 → 0, 그다음 완전 중복 행 제거 (`noduprecs`)
- Python 1.3 대응: `fillna(0)` 후 `drop_duplicates()` — 채우기와 중복 제거의 순서가 같아야 결과가 같다
- `COALESCE(열, 0)`: 값이 NULL이면 0, 아니면 원래 값 그대로
- `SELECT DISTINCT`: 모든 열 값이 완전히 같은 행은 하나만 남긴다

In [4]:
conn.execute("DROP TABLE IF EXISTS fire")
conn.execute("""
CREATE TABLE fire AS
SELECT DISTINCT
       FIRE_DT, SIDO, SIGUNGU, FIRE_TYPE, CAUSE_L, CAUSE_S,
       CASUALTY, DEATH, INJURY, DMG_TOTAL,
       COALESCE(DMG_REAL, 0) AS DMG_REAL,
       COALESCE(DMG_MOVABLE, 0) AS DMG_MOVABLE,
       PLACE_L, PLACE_M, PLACE_S, FIRE_MONTH
FROM fire_raw
""")
conn.commit()

#### 4.2-2 시군구 단위 집계 → fire_agg 테이블
- Python 1.4 `groupby(['시도','시군구']).agg(...)`, SAS 41~50행 PROC SQL 대응
- 시도+시군구 두 열로 묶어 화재건수·사망·부상·재산피해 합계를 구한다

In [5]:
conn.execute("DROP TABLE IF EXISTS fire_agg")
conn.execute("""
CREATE TABLE fire_agg AS
SELECT SIDO, SIGUNGU,
       COUNT(*) AS FIRE_CNT,
       COUNT(DEATH) AS DEATH_CNT,
       SUM(INJURY) AS INJURY_CNT,
       SUM(DMG_TOTAL) AS DMG_TOTAL_SUM
FROM fire
GROUP BY SIDO, SIGUNGU
""")
conn.commit()

#### 4.2-3 정제·집계 결과 행 수 확인
- 원자료 → 정제 후 → 시군구 집계로 넘어가며 행 수가 어떻게 바뀌는지 한 번에 확인

In [6]:
q = """
SELECT (SELECT COUNT(*) FROM fire_raw) AS 원자료_행수,
       (SELECT COUNT(*) FROM fire)     AS 정제후_행수,
       COUNT(*)                        AS 시군구수,
       SUM(FIRE_CNT)                   AS 화재건수_합계
FROM fire_agg
"""
pd.read_sql(q, conn)

,원자료_행수,정제후_행수,시군구수,화재건수_합계
0,191509,191509,254,191509


> **정제 전후 191,509행으로 동일 → 결측을 0으로 채운 뒤 새로 생긴 중복 행 없음** <br>
> 시군구 254개, 화재건수 합계 191,509 — 노트북 1.4(254)·5.1(39,462 + 152,047)과 일치. <br>
> 원자료 191,510행(노트북 1.1)과 비교하면, 1행은 CSV로 저장하기 전 Python 단계에서 이미 제거됨.

### 4.3 인구 테이블과 JOIN
- JOIN: 두 테이블을 공통 열(열쇠, key)로 짝지어 옆으로 붙이는 것. pandas `merge`, SAS `proc sql`의 `join` 대응
- 이번 열쇠: SIDO + SIGUNGU 두 열 (시군구 이름만으로는 겹치는 지역이 있음 — HIRA 고성군)
- INNER JOIN: 양쪽 테이블에 모두 있는 행만 남긴다
- LEFT JOIN: 왼쪽 테이블의 행은 전부 남기고, 오른쪽에 짝이 없으면 오른쪽 열을 NULL로 채운다
#### 4.3-1 JOIN 종류별 결과 행 수 비교
- 같은 두 테이블을 INNER / 화재 기준 LEFT / 인구 기준 LEFT로 붙였을 때 행 수가 어떻게 다른지 확인

In [7]:
q = """
SELECT 
   (SELECT COUNT(*)
   FROM fire_agg AS a INNER JOIN pop AS b
   on a.SIDO = b.SIDO AND a.SIGUNGU = b.SIGUNGU) AS INNER_JOIN,
   (SELECT COUNT(*)
   FROM fire_agg AS a LEFT JOIN pop AS b
   on a.SIDO = b.SIDO AND a.SIGUNGU = b.SIGUNGU) AS 화재기준_LEFT,
   (SELECT COUNT(*)
   FROM pop AS b LEFT JOIN fire_agg AS a
   ON a.SIDO = b.SIDO AND a.SIGUNGU = b.SIGUNGU) AS 인구기준_LEFT
"""
pd.read_sql(q, conn)

,INNER_JOIN,화재기준_LEFT,인구기준_LEFT
0,254,254,265


> **JOIN 종류별 행 수: INNER 254, 화재 기준 LEFT 254, 인구 기준 LEFT 265** <br>
> INNER와 화재 기준 LEFT가 같음 → 화재 쪽 254개 시군구는 모두 인구 테이블에 짝이 있음. <br>
> 인구 기준 LEFT − INNER = 11 → 인구 테이블에만 있는 지역 11개 (4.3-4).

#### 4.3-2 화재 집계 기준 LEFT JOIN + 10만 명당 발생율 → analysis 테이블
- Python 1.5 `merge(how='left')`, SAS 63~71행 `left join` 대응
- 화재 집계(254개)를 왼쪽에 두어, 화재 쪽 시군구가 하나도 빠지지 않게 한다
- 발생율 = 화재건수 ÷ 5년 평균인구 × 100,000 (소수 둘째 자리 반올림)

In [8]:
conn.execute("DROP TABLE IF EXISTS analysis")
conn.execute("""
CREATE TABLE analysis AS
SELECT a.SIDO, a.SIGUNGU,
       a.FIRE_CNT, a.DEATH_CNT, a.INJURY_CNT, a.DMG_TOTAL_SUM,
       b.POP_AVG_5YR,
       ROUND(a.FIRE_CNT / b.POP_AVG_5YR * 100000, 2) AS FIRE_RATE_100K
FROM fire_agg AS a
LEFT JOIN pop AS b
 ON a.SIDO = b.SIDO AND a.SIGUNGU = b.SIGUNGU
""")
conn.commit()

#### 4.3-3 병합 누락 확인
- Python 1.5 `isna().sum()`, SAS `proc means nmiss` 대응
- LEFT JOIN에서 짝을 못 찾은 행은 인구 열이 NULL → `IS NULL`로 센다

In [9]:
q = """
SELECT COUNT(*) AS 인구_누락
FROM analysis
WHERE POP_AVG_5YR IS NULL
"""
pd.read_sql(q, conn)

,인구_누락
0,0


> **병합 누락 0 — 노트북 1.5와 일치**

#### 4.3-4 인구 테이블에만 있는 지역 찾기
- 방향을 바꿔 인구를 왼쪽에 두고 LEFT JOIN → 화재 쪽 열이 NULL인 행 = 화재 집계에는 없는 지역
- Python·SAS에서는 확인하지 않았던 부분 (두 테이블의 범위 차이)

In [10]:
q = """
SELECT b.SIDO, b.SIGUNGU, b.POP_AVG_5YR
FROM pop AS b
LEFT JOIN fire_agg AS a
 ON a.SIDO = b.SIDO AND a.SIGUNGU = b.SIGUNGU
WHERE a.SIGUNGU IS NULL
"""
pd.read_sql(q, conn)

,SIDO,SIGUNGU,POP_AVG_5YR
0,경기도,고양시,1076426.18
1,경기도,성남시,927492.55
2,경기도,수원시,1190420.65
3,경기도,안산시,643211.93
4,경기도,안양시,550110.97
5,경기도,용인시,1076679.67
6,경상남도,창원시,1023702.27
7,경상북도,포항시,498747.18
8,전북특별자치도,전주시,650616.95
9,충청남도,천안시,657549.62


> **인구 테이블에만 있는 지역 11개 — 모두 '○○시'** <br>
> 고양시·성남시·수원시·안산시·안양시·용인시(경기), 창원시(경남), 포항시(경북), 전주시(전북), 천안시(충남), 청주시(충북). <br>
> 4.3-1의 인구 기준 LEFT − INNER(265 − 254 = 11)와 같은 수.

#### 4.3-5 인구 테이블에만 있는 시 확인 (고양시 예시)
- 4.3-4에서 인구에만 있는 11개 지역이 모두 "○○시" → 시 전체 행과 구 행이 함께 있는지 확인
- 인구·화재 두 테이블에서 이름이 '고양시'로 시작하는 행을 위아래로 이어 붙여 비교한다
- `UNION ALL`: 두 SELECT 결과를 위아래로 이어 붙인다 (JOIN은 옆으로 붙이기)

In [16]:
q = """
SELECT '인구' AS 출처, SIGUNGU, POP_AVG_5YR AS 값
FROM pop
WHERE SIDO = '경기도' AND SIGUNGU LIKE '고양시%'
UNION ALL
SELECT '화재', SIGUNGU, FIRE_CNT
FROM fire_agg
WHERE SIDO = '경기도' AND SIGUNGU LIKE '고양시%'
ORDER BY 출처, SIGUNGU
"""
pd.read_sql(q, conn)

,출처,SIGUNGU,값
0,인구,고양시,1076426.18
1,인구,고양시덕양구,485832.45
2,인구,고양시일산동구,296726.55
3,인구,고양시일산서구,293867.18
4,화재,고양시덕양구,1031.00
5,화재,고양시일산동구,725.00
6,화재,고양시일산서구,459.00


> **고양시: 인구 파일에는 '시 전체' 행과 '구' 행이 함께 있고, 화재 데이터는 구 단위만 있음** <br>
> 인구: 고양시 1,076,426.18 / 덕양구 485,832.45 · 일산동구 296,726.55 · 일산서구 293,867.18 — 구 3개의 합이 시 전체와 같음. <br>
> 화재: 덕양구 1,031 · 일산동구 725 · 일산서구 459 (시 전체 행 없음). <br>
> → 화재 기준 LEFT JOIN이라 시 전체 행은 결과에 들어가지 않아 인구가 이중으로 잡히지 않음. 인구 테이블을 시도별로 합산하면 이중 계산될 수 있으므로 주의. (나머지 10개 시는 확인하지 않음)

### 4.4 결과 확인
#### 4.4-1 화재발생율 상위 5개 시군구
- Python 1.5 `sort_values('화재발생율_10만명당', ascending=False).head()` 대응

In [11]:
q = """
SELECT SIDO, SIGUNGU, FIRE_CNT, POP_AVG_5YR, FIRE_RATE_100K
FROM analysis
ORDER BY FIRE_RATE_100K DESC
LIMIT 5
"""
pd.read_sql(q, conn)

,SIDO,SIGUNGU,FIRE_CNT,POP_AVG_5YR,FIRE_RATE_100K
0,경상남도,의령군,428,26143.18,1637.14
1,전라남도,함평군,502,31180.20,1610.00
2,경상북도,고령군,455,30683.92,1482.86
3,경상남도,산청군,504,34264.27,1470.92
4,전라남도,영암군,774,52905.27,1462.99


> **발생율 상위 5개 — 노트북 1.5와 건수·인구·발생율 모두 일치** <br>
> 의령군 1,637.14 · 함평군 1,610.00 · 고령군 1,482.86 · 산청군 1,470.92 · 영암군 1,462.99. <br>
> 나누는 쪽(5년 평균인구)이 소수형이라 CAST 없이도 소수 나눗셈이 됨을 확인.

#### 4.4-2 Python 결과 파일을 테이블로 불러오기
- Python 1.5에서 저장한 fire_population_analysis.csv(254행) → `py_result` 테이블
- 다음 셀에서 SQL 결과와 JOIN해 전체 대조한다

In [12]:
py_result = pd.read_csv(r'C:\data\fire_population_analysis.csv', encoding='utf-8-sig')
py_result.to_sql('py_result', conn, if_exists='replace', index=False)

254

#### 4.4-3 SQL 결과 vs Python 결과 254행 전체 대조
- 두 결과를 SIDO + SIGUNGU로 INNER JOIN한 뒤, 열마다 값이 다른 행 수를 센다
- SQL_02 2.4-4의 pandas 대조(`!=` + `.sum()`)를 SQL로 옮긴 것

In [13]:
q = """
SELECT COUNT(*)                                    AS 대조_행수,
       SUM(a.FIRE_CNT       <> p.FIRE_CNT)         AS FIRE_CNT_차이,
       SUM(a.DEATH_CNT      <> p.DEATH_CNT)        AS DEATH_CNT_차이,
       SUM(a.INJURY_CNT     <> p.INJURY_CNT)       AS INJURY_CNT_차이,
       SUM(a.DMG_TOTAL_SUM  <> p.DMG_TOTAL_SUM)    AS DMG_차이,
       SUM(a.POP_AVG_5YR    <> p.POP_AVG_5YR)      AS POP_차이,
       SUM(a.FIRE_RATE_100K <> p.FIRE_RATE_100K)   AS RATE_차이
FROM analysis AS a
INNER JOIN py_result AS p
  ON a.SIDO = p.SIDO AND a.SIGUNGU = p.SIGUNGU
"""
pd.read_sql(q, conn)

,대조_행수,FIRE_CNT_차이,DEATH_CNT_차이,INJURY_CNT_차이,DMG_차이,POP_차이,RATE_차이
0,254,0,254,0,0,0,0


> **Python 결과와 254행 대조 — 5개 열 차이 0, DEATH_CNT만 254행 모두 다름** <br>
> FIRE_CNT·INJURY_CNT·DMG_TOTAL_SUM·POP_AVG_5YR·FIRE_RATE_100K는 254행 모두 일치. <br>
> DEATH_CNT 차이 원인은 4.4-4에서 확인.

#### 4.4-4 DEATH_CNT 차이 원인 확인
- 4.4-3에서 DEATH_CNT만 254행 모두 SQL과 Python 값이 다름
- 노트북 1.5에 출력된 상위 5개 지역의 사망·부상 수와 SQL·Python 값을 나란히 놓고 비교한다

In [15]:
q = """
SELECT a.SIDO, a.SIGUNGU,
       a.DEATH_CNT  AS SQL_사망,  p.DEATH_CNT  AS PY_사망,
       a.INJURY_CNT AS SQL_부상,  p.INJURY_CNT AS PY_부상
FROM analysis AS a
INNER JOIN py_result AS p
  ON a.SIDO = p.SIDO AND a.SIGUNGU = p.SIGUNGU
WHERE a.SIGUNGU IN ('의령군', '함평군', '고령군', '산청군', '영암군')
"""
pd.read_sql(q, conn)

,SIDO,SIGUNGU,SQL_사망,PY_사망,SQL_부상,PY_부상
0,경상남도,산청군,504,4,21,21
1,경상남도,의령군,428,3,13,13
2,경상북도,고령군,455,1,18,18
3,전라남도,영암군,774,5,15,15
4,전라남도,함평군,502,4,10,10


> **SQL 쪽 DEATH_CNT에 화재건수가 들어가 있음 → SQL 테이블 생성 과정의 입력 문제** <br>
> 상위 5개 지역의 SQL_사망(504·428·455·774·502)이 4.4-1의 화재건수와 같음. <br>
> PY_사망(4·3·1·5·4)과 부상 수(SQL·PY 모두 21·13·18·15·10)는 노트북 1.5 출력과 일치. <br>